In [ ]:
!pip install -q transformers
!pip install -q torch

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

import torch
import torch.nn.functional as F

In [ ]:
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

print("FinBERT loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT loaded successfully


In [ ]:
print(model.config.id2label)

{0: 'positive', 1: 'negative', 2: 'neutral'}


In [ ]:
sentences = [

    # Positive
    "The company reported a 35 percent increase in quarterly profits and exceeded analyst expectations.",

    # Positive
    "Revenue grew strongly while operating margins improved significantly.",

    # Negative
    "The rating agency issued a credit downgrade warning due to rising debt levels.",

    # Negative
    "The company expects substantial losses and declining cash flow next quarter.",

    # Neutral
    "The Reserve Bank of India announced its monetary policy decision today."
]

In [ ]:
def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    probabilities = F.softmax(
        logits,
        dim=-1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    label = model.config.id2label[predicted_class]

    confidence = probabilities[0][predicted_class].item()

    return label, confidence

In [ ]:
for sentence in sentences:

    label, confidence = predict_sentiment(
        sentence
    )

    print("=" * 80)

    print("Sentence:")
    print(sentence)

    print("\nPrediction:")
    print(label)

    print("Confidence:")
    print(round(confidence, 4))

Sentence:
The company reported a 35 percent increase in quarterly profits and exceeded analyst expectations.

Prediction:
positive
Confidence:
0.9539
Sentence:
Revenue grew strongly while operating margins improved significantly.

Prediction:
positive
Confidence:
0.9611
Sentence:
The rating agency issued a credit downgrade warning due to rising debt levels.

Prediction:
negative
Confidence:
0.9686
Sentence:
The company expects substantial losses and declining cash flow next quarter.

Prediction:
negative
Confidence:
0.9722
Sentence:
The Reserve Bank of India announced its monetary policy decision today.

Prediction:
neutral
Confidence:
0.8786


In [ ]:
for sentence in sentences:

    inputs = tokenizer(
        sentence,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(
        outputs.logits,
        dim=-1
    )

    print("\n")
    print(sentence)

    for idx, prob in enumerate(probs[0]):

        label = model.config.id2label[idx]

        print(
            f"{label}: {prob:.4f}"
        )



The company reported a 35 percent increase in quarterly profits and exceeded analyst expectations.
positive: 0.9539
negative: 0.0232
neutral: 0.0229


Revenue grew strongly while operating margins improved significantly.
positive: 0.9611
negative: 0.0184
neutral: 0.0205


The rating agency issued a credit downgrade warning due to rising debt levels.
positive: 0.0094
negative: 0.9686
neutral: 0.0220


The company expects substantial losses and declining cash flow next quarter.
positive: 0.0070
negative: 0.9722
neutral: 0.0208


The Reserve Bank of India announced its monetary policy decision today.
positive: 0.0758
negative: 0.0456
neutral: 0.8786


In [ ]:
test_sentences = [

    "Profits increased but future demand remains uncertain.",

    "The merger could create opportunities but may also increase risk.",

    "The RBI maintained interest rates as expected.",

    "The company beat earnings estimates despite weak revenue growth."
]

In [ ]:
for s in test_sentences:

    label, confidence = predict_sentiment(s)

    print("\n")
    print(s)

    print(label)

    print(round(confidence,4))



Profits increased but future demand remains uncertain.
positive
0.9514


The merger could create opportunities but may also increase risk.
positive
0.9151


The RBI maintained interest rates as expected.
neutral
0.836


The company beat earnings estimates despite weak revenue growth.
positive
0.7668
